# litert

> A fastllm-style `Chat` over litert_lm - message helpers, an ordered callback system, human-in-the-loop tool approval, sync streaming, and `token_count`-based usage tracking.

`rishi.litert` wraps Google's on-device [litert_lm](https://github.com/google-ai-edge/litert-lm) engine in a small, callable `Chat`. It runs Gemma models locally (CPU/GPU), keeps a Python-visible message history, streams output in notebooks, tracks token usage so you know when to compress, and lets you gate tool calls behind an approval function.

In [ ]:
#| default_exp litert

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, re, os, asyncio, io, base64, uuid
from html import escape
from mimetypes import guess_type
from contextlib import ExitStack, redirect_stdout
from litert_lm import (Engine, Backend, Conversation, Session, Message, Contents, Content, Role, ToolCall,
                       ToolEventHandler, SamplerConfig, Benchmark, set_min_log_severity)
from litert_lm._messages import Text, ImageBytes, ImageFile, AudioBytes, AudioFile, ToolResponse, normalize_message
from huggingface_hub import hf_hub_download, list_repo_files, scan_cache_dir
from fastcore.all import Path, store_attr, patch, L, GetAttr, ifnone, detect_mime, first, listify, img_bytes, AttrDict, in_
from safepyrun import RunPython
from rishi import core
from rishi.core import *


/Users/71293/code/personal/orgs/rishi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Messages

litert speaks a small message schema (`Message`, `Contents`, and `Content` subtypes). These helpers build them from ordinary Python values.

- `mk_content` maps one value to a litert `Content`: a `str` becomes text, `bytes` become an image or audio (sniffed with `detect_mime`), and a `Path` becomes an image or audio file reference (by MIME). An existing `Content` passes through.
- `mk_msg` wraps content into a `Message` (default role `user`), and also accepts an existing `Message` or a `{'role','content'}` dict.
- `mk_msgs` normalises a mixed list into canonical litert message dicts, used to seed a conversation's history.

In [ ]:
#| export

def _mk_content(o):
    'Convert `o` to a litert `Content`, sniffing bytes/files for image vs audio.'
    if isinstance(o, Content): return o
    if isinstance(o, str): return Text(o)
    if isinstance(o, bytes): return AudioBytes(o) if (detect_mime(o) or '').startswith('audio/') else ImageBytes(o)
    if isinstance(o, Path): return AudioFile(str(o)) if (guess_type(str(o))[0] or '').startswith('audio/') else ImageFile(str(o))
    raise TypeError(f"Unsupported content type: {type(o)}")

def _mk_msg(content, role='user'):
    'Create a litert `Message` from str/bytes/list/dict/Message.'
    if content is None: return None
    if isinstance(content, Message): return content
    if isinstance(content, dict): return Message(Role(content['role']), Contents.of(content['content']))
    parts = [_mk_content(o) for o in content] if isinstance(content, list) else [_mk_content(content)]
    return Message(Role(role), Contents(parts))

def _mk_msgs(msgs):
    'Normalize a list of messages to litert message dicts.'
    if not msgs: return []
    return [normalize_message(m if isinstance(m, (Message, dict)) else _mk_msg(m)) for m in listify(msgs)]

what does mk_content do?

In [ ]:
_mk_content("hello\n").text

'hello\n'

In [ ]:
from fastcore.test import test_eq, test_fail

In [ ]:
test_eq(_mk_msg("hello").to_json(), {"role": "user", "content": [{"type": "text", "text": "hello"}]})
test_eq(_mk_msg("hi", role="model").to_json()["role"], "model")
test_eq([x["role"] for x in _mk_msgs(["a", _mk_msg("b", role="model")])], ["user", "model"])
assert isinstance(_mk_content("x"), Text)
# bytes are sniffed: image vs audio
assert isinstance(_mk_content(b'\x89PNG\r\n\x1a\n' + b'\x00'*16), ImageBytes)
assert isinstance(_mk_content(b'RIFF\x00\x00\x00\x00WAVE' + b'\x00'*16), AudioBytes)

## Portable history

`chat.hist` is kept in one backend-agnostic **canonical** shape (OpenAI-style dicts: `assistant`/`user`/`system`/`tool` roles, string content, `channels.thought`, and `tool_calls`/`tool_call_id` for tool rounds) so a conversation can be handed from one backend to another. litert's own wire format differs (`model` role, list content, `image`/`audio` blobs, `tool_response` parts, id-less tool calls), so two converters bridge it:

- `fmt2hist` — litert-native messages (or a `Message`/`Resp`) → canonical history dicts; used to record turns into `chat.hist`. Past-turn media collapses to an `[image]`/`[audio]` placeholder.
- `hist2fmt` — canonical history dicts → litert `Message`s the engine can ingest; used to seed a conversation from a (possibly ported) history.

Because llama's history is already canonical, `Chat('litert/…', messages=llama_chat.hist)` and `Chat('llama/…', messages=litert_chat.hist)` both work — see the port example in the [overview](index.html).

In [ ]:
#| export
def _call_id(): return f"call_{uuid.uuid4().hex[:8]}"

_ph = {'image': '[image]', 'audio': '[audio]'}
def _canon_content(c):
    "litert content (str or list of parts) -> a canonical string; media -> `[image]`/`[audio]`."
    if c is None or isinstance(c, str): return c
    return '\n'.join(p.get('text', '') if p.get('type') == 'text' else _ph.get(p.get('type'), '[media]')
                     for p in c if isinstance(p, dict))

def _fmt2hist(m, dflt_role='user'):
    "One litert message (str/`Message`/native dict/`Resp`) -> a canonical rishi history dict."
    if isinstance(m, str): return {'role': 'user', 'content': m}
    if isinstance(m, Message): m = m.to_json()
    elif not isinstance(m, dict): m = _mk_msg(m).to_json()
    role = 'assistant' if m.get('role') == 'model' else m.get('role', dflt_role)
    content = m.get('content')
    if role == 'tool':
        if isinstance(content, list):
            tr = first(content, lambda p: isinstance(p, dict) and p.get('type') == 'tool_response') or {}
            out = {'role': 'tool', 'name': tr.get('name', ''), 'content': str(tr.get('response', ''))}
        else: out = {'role': 'tool', 'content': content if isinstance(content, str) else str(content)}
        if m.get('name'): out['name'] = m['name']
        if m.get('tool_call_id'): out['tool_call_id'] = m['tool_call_id']
        return out
    out = {'role': role}
    if (cc := _canon_content(content)) is not None: out['content'] = cc
    if m.get('channels'): out['channels'] = dict(m['channels'])
    if m.get('tool_calls'):
        out['tool_calls'] = [{'id': tc.get('id') or _call_id(), 'type': 'function',
                              'function': {'name': tc.get('function', {}).get('name', ''),
                                           'arguments': tc.get('function', {}).get('arguments', {})}}
                             for tc in m['tool_calls']]
        out.setdefault('content', '')
    return out

def _data_uri_bytes(url):
    "Decode a `data:...;base64,...` URI to bytes, else None."
    if isinstance(url, str) and url.startswith('data:') and ';base64,' in url:
        return base64.b64decode(url.split(';base64,', 1)[1])
    return None

def _litert_contents(c):
    "Canonical content (str or OpenAI-style parts) -> a litert `Contents`."
    if c is None: return Contents.empty()
    if isinstance(c, str): return Contents.of(c)
    parts = []
    for p in c:
        if not isinstance(p, dict): continue
        t = p.get('type')
        if t == 'text': parts.append(Text(p.get('text', '')))
        elif t == 'image_url':
            u = p.get('image_url', {}); url = u.get('url') if isinstance(u, dict) else u
            parts.append(ImageBytes(b) if (b := _data_uri_bytes(url)) is not None else ImageFile(url))
        elif t == 'input_audio': parts.append(AudioBytes(base64.b64decode((p.get('input_audio') or {}).get('data', ''))))
    return Contents(parts)

def _hist2fmt1(m):
    "One canonical rishi history dict (or raw input) -> a litert `Message` for the engine."
    if isinstance(m, Message): return m
    if isinstance(m, str): return Message.user(m)
    if not isinstance(m, dict): return _mk_msg(m)
    role = m.get('role', 'user')
    if role == 'tool': return Message.tool(Contents([ToolResponse(m.get('name', ''), m.get('content', ''))]))
    tcs = [ToolCall(tc.get('function', {}).get('name', ''), tc.get('function', {}).get('arguments', {}))
           for tc in (m.get('tool_calls') or [])]
    return Message(Role('model' if role == 'assistant' else role), _litert_contents(m.get('content')),
                   tool_calls=tcs, channels=(m.get('channels') or None))

def _to_litert_msg(m):
    "Raw input or a (canonical/native) chat dict -> a litert `Message` for the engine preface."
    return m if isinstance(m, Message) else (_hist2fmt1(m) if isinstance(m, dict) else _mk_msg(m))


## Usage tracking

litert doesn't return per-response token counts, but `conv.token_count` exposes the running KV-cache size (prefill plus decode). `UsageStats` records one turn's `prompt`, `completion`, and `total` tokens, derived from the count delta around the turn, and adds across turns. Compare `chat.token_count` (or `chat.pct_full`) against `ctx_limit` to decide when to compress the conversation.

## Callbacks

A small, ordered callback system inspired by fastllm. A `ChatCallback` subclass hooks named events (`after_msgs`, `before_send`, `after_response`, `before_tool_calls`, `after_tool_calls`) and reads live turn state off the chat via `GetAttr`, so `self.turn_msg` is `chat.turn_msg`. `run_cbs` dispatches one event to every enabled callback in `order`, forwarding anything a callback yields into the output stream.

## Streaming display

`StreamFormatter` turns a litert response stream into markdown as it arrives: text passes through, and tool calls render as a compact `⏳ name(args)` line. `display_stream` consumes a markdown-chunk stream (what `chat(msg, stream=True)` yields) and renders it live in a notebook via IPython. `mk_tr_details` formats a completed tool call as a collapsible JSON block.

## Built-in callbacks

Three callbacks make up `_dflt_cbs` and run on every chat unless you pass `default_cbs=False`. `HistoryCallback` records each outgoing message and reply into `chat.hist`. `UsageCallback` folds the turn's token counts into `chat.use`, read from a `conv.token_count` delta. `ToolReminderCallback` appends a short reminder to outgoing messages, but only when the chat has tools, nudging the model to summarise tool results in prose before it continues.

Order matters. `HistoryCallback` and `UsageCallback` sit at the front (low `order`) so a turn is recorded before feature callbacks like `PyFenceCallback` react to it and feed messages back. A callback reads live turn state off the chat through `GetAttr`, so `self.turn_res` is `chat.turn_res`.

In [ ]:
#| export
class _ToolReminderCallback(ChatCallback):
    'Inject a tool-summary reminder into the outgoing message when tools are registered.'
    order = 30
    def __init__(self, tool_reminder=tool_reminder_): store_attr()
    def before_send(self):
        if self.chat.tools and self.chat.turn_msg is not None: self.chat.turn_msg.contents.contents.append(Text(self.tool_reminder))


In [ ]:
#| export
class _HistoryCallback(ChatCallback):
    'Record the outgoing message and the response into `chat.hist` (canonical form).'
    order = 0
    def before_send(self):
        if self.chat.turn_msg is not None:
            self.chat.hist.append(_fmt2hist(self.chat.turn_msg))
            self.chat._turn_msg_recorded = True
    def after_response(self): self.chat.hist.append(_fmt2hist(self.chat.turn_res, 'assistant'))

class _UsageCallback(ChatCallback):
    'Fold each response\'s token usage into `chat.use` from a `token_count` diff.'
    order = 10
    def after_response(self):
        c = self.chat; delta = c.conv.token_count - c._tc0
        out = len(c.engine.tokenize(resp_text(c.turn_res)))
        new_prompt, cached = max(delta - out, 0), c._tc0
        c.use += UsageStats(prompt_tokens=cached + new_prompt, completion_tokens=out,
                            total_tokens=cached + delta, n=1, cached_tokens=cached)


## Tool calling, approval, and history

litert runs the tool-call loop inside the engine. `ChatToolHandler` bridges that loop back to Python: for each call it records the request and result into `chat.hist` in canonical form (a paired assistant `tool_calls` entry and a `tool` result, so the round survives a hand-off to another backend), fires the `before_tool_calls` and `after_tool_calls` callbacks, and consults `chat.approve(tool_call)` before executing. Returning `False` blocks the tool and feeds a "Denied by human operator" response back to the model. That is the hook for human-in-the-loop gating, shown below.

In [ ]:
#| export
def _tc_name(tc): return tc.get('function', {}).get('name', '')

class ChatToolHandler(ToolEventHandler):
    "Bridge litert's in-engine tool loop to Chat callbacks, HITL approval, the tool-call budget, and history."
    def __init__(self, chat): self.chat = chat
    def approve_tool_call(self, tool_call):
        c = self.chat
        c.turn_tc = tool_call
        for _ in run_cbs(c, 'before_tool_calls'): pass
        over = c._budget_exceeded or (c.max_steps is not None and c._steps >= c.max_steps)
        ok = False if over else (c.approve(tool_call) if c.approve else True)
        if ok: c._steps += 1
        else: c._budget_exceeded = c._budget_exceeded or over
        fn = tool_call.get('function', {}); self._tcid = _call_id()
        c.hist.append({'role': 'assistant', 'content': '',
            'tool_calls': [{'id': self._tcid, 'type': 'function',
                            'function': {'name': fn.get('name', ''), 'arguments': fn.get('arguments', {})}}]})
        if not ok: c.hist.append({'role': 'tool', 'tool_call_id': self._tcid, 'name': fn.get('name', ''),
                                  'content': budget_msg_ if over else 'Denied by human operator'})
        return ok
    def process_tool_response(self, tool_response):
        mx = getattr(self.chat, 'tool_max_len', None)
        if mx and isinstance(tool_response, str) and len(tool_response) > mx:
            tool_response = tool_response[:mx] + ' …[truncated]'
        self.chat.turn_tool_result = tool_response
        self.chat.hist.append({'role': 'tool', 'tool_call_id': getattr(self, '_tcid', None),
                               'name': _tc_name(self.chat.turn_tc), 'content': str(tool_response)})
        for _ in run_cbs(self.chat, 'after_tool_calls'): pass
        return tool_response

In [ ]:
#| hide
from fastcore.test import test_eq

class _FakeChat:
    "Just the attributes `ChatToolHandler` touches - enough to drive it without an engine."
    def __init__(self, approve=None, max_steps=None):
        self.hist, self.cbs, self.approve, self.tool_max_len = [], L(), approve, None
        self.max_steps, self._steps, self._budget_exceeded = max_steps, 0, False

tc = {'function': {'name': 'add', 'arguments': {'a': 1, 'b': 2}}}

ch = _FakeChat(); h = ChatToolHandler(ch)
test_eq(h.approve_tool_call(tc), True)
test_eq(ch.hist[-1]['role'], 'assistant')
test_eq(h.process_tool_response('3'), '3')
test_eq(ch.hist[-1]['role'], 'tool'); test_eq(ch.hist[-1]['content'], '3')

# HITL: `approve` can deny, and the model is told
ch = _FakeChat(approve=lambda tc: False); h = ChatToolHandler(ch)
test_eq(h.approve_tool_call(tc), False)
test_eq(ch.hist[-1]['content'], 'Denied by human operator')

# budget: past `max_steps` calls, further ones are denied whatever `approve` says
ch = _FakeChat(max_steps=1); h = ChatToolHandler(ch)
test_eq(h.approve_tool_call(tc), True)
test_eq(h.approve_tool_call(tc), False)
test_eq(ch.hist[-1]['content'], budget_msg_)
assert ch._budget_exceeded          # `Chat.__call__` now sends `final_prompt` to close the turn out

# max_steps=None means no cap
ch = _FakeChat(); h = ChatToolHandler(ch)
assert all(h.approve_tool_call(tc) for _ in range(20))

## Loading models & Chat

Default litert-community Gemma repos live at [huggingface.co/litert-community](https://huggingface.co/litert-community). `get_model` resolves a `.litertlm` file with a cache-first ladder: an explicit `model_path`, then the local HuggingFace cache (`scan_cache_dir`, no network), then a download. It prefers the native build over `-web` variants, which omit the CPU/GPU decode graph.

`Chat` ties this together into a callable. Build it (it constructs or accepts an `Engine`), then call it like a function: one turn per call, updating history and usage. `create_engine` is a patchable classmethod that builds the engine and creates `cache_dir` if given. Pass a prebuilt `engine=` to share one model across several chats. Engine and conversation are entered on an `ExitStack`, so `close()` releases them in the right order and never closes an engine you supplied.

In [ ]:
#| export
gemma4_e4b='litert-community/gemma-4-E4B-it-litert-lm'
gemma4_e2b='litert-community/gemma-4-E2B-it-litert-lm'
gemma4_12b='litert-community/gemma-4-12B-it-litert-lm'

In [ ]:
#| export
def _litertlm(fs):
    "First native `.litertlm` path in `fs` (skips `-web`/other builds)."
    return first(fs, lambda p: p.endswith('.litertlm') and 'web' not in p)

def _cached_model(model_id):
    "Local `.litertlm` path from the HF cache without hitting the network, else None."
    try: repo = first(scan_cache_dir().repos, lambda r: r.repo_id == model_id)
    except Exception: return None
    return _litertlm(str(f.file_path) for r in repo.revisions for f in r.files) if repo else None

def _get_model(model_id, model_path=None):
    "Return a local `.litertlm` path: `model_path`, else HF cache, else download."
    if model_path and Path(model_path).exists(): return model_path
    if (hit := _cached_model(model_id)): return hit
    if not (fn := _litertlm(list_repo_files(model_id))): raise FileNotFoundError(f"No .litertlm file found for {model_id}")
    return hf_hub_download(model_id, fn)

def _merge_chunks(chunks):
    "Reconstruct an assistant response dict (text + thinking) from streamed litert chunks."
    text, th = ''.join(resp_text(c) for c in chunks), ''.join(thought(c) for c in chunks)
    r = {'role': 'assistant', 'content': [{'type': 'text', 'text': text}]}
    if th: r['channels'] = {'thought': th}
    return Resp(r)

_dflt_cbs = [_HistoryCallback, _UsageCallback, _ToolReminderCallback]

class LitertChat(core.Chat):
    "Sync chat over a local litert_lm engine."
    _runtime = 'litert'
    _dflt_cbs = _dflt_cbs
    mk_content, mk_msg, mk_msgs = staticmethod(_mk_content), staticmethod(_mk_msg), staticmethod(_mk_msgs)

    @staticmethod
    def fmt2hist(msgs):
        "litert-native messages -> canonical rishi history dicts."
        return [_fmt2hist(m) for m in listify(msgs)]
    @staticmethod
    def hist2fmt(msgs):
        "Canonical rishi history dicts -> litert `Message`s for the engine."
        return [_hist2fmt1(m) for m in listify(msgs)]

    @classmethod
    def create_engine(cls, model_id=gemma4_e2b, model_path=None, be=Backend.CPU(), vbe=Backend.CPU(),
                      abe=Backend.CPU(), multimodal=True, cache_dir=None, enable_speculative_decoding=None, **kw):
        'Build a litert `Engine`; creates `cache_dir` if given. Override/`@patch` to customize.'
        if cache_dir: Path(cache_dir).mkdir(parents=True, exist_ok=True)
        mod, mm = _get_model(model_id, model_path), dict(vision_backend=vbe, audio_backend=abe) if multimodal else {}
        return Engine(mod, backend=be, cache_dir=cache_dir or '', enable_speculative_decoding=enable_speculative_decoding, **mm, **kw)

    def __init__(self, model=None, *, runtime=None, model_path=None, engine=None, backend=Backend.CPU(),
                 multimodal=True, cache_dir=None, enable_speculative_decoding=None, eng_kw=None,
                 sp='', messages=None, tools=None, ctx_limit=None, approve=None, tool_max_len=None,
                 max_steps=10, final_prompt=dflt_final_prompt_, parallel_tools=False,
                 think=False, filter_think=True, temp=None, top_k=None, top_p=None,
                 seed=None, sampler_config=None, max_output_tokens=None, conv_kw=None, cbs=None, default_cbs=True):
        if parallel_tools: raise NotImplementedError(
            "litert runs its tool loop inside the engine, so it can't dispatch calls in parallel; "
            "use runtime='llama' (or 'mlx') for parallel_tools=True.")
        model = core.split_runtime(model)[1]
        model_id = None if model is None or core._is_path(model) else model
        model_path = model_path or (model if model and core._is_path(model) else None)
        self._stack, self._conv_stack = ExitStack(), ExitStack()
        self._own_engine = engine is None
        if self._own_engine:
            engine = self.create_engine(model_id or gemma4_e2b, model_path, backend,
                multimodal=multimodal, cache_dir=cache_dir, enable_speculative_decoding=enable_speculative_decoding, **(eng_kw or {}))
            self.engine = self._stack.enter_context(engine)
        else: self.engine = engine
        # the system prompt is kept apart from `hist` and re-applied on every rebuild, so eviction can
        # never drop it - it is the anchor a sliding window is supposed to preserve
        self._sys_pre = [{'role': 'system', 'content': sp}] if sp else []
        if sampler_config is None and any(x is not None for x in (temp, top_k, top_p, seed)):
            sampler_config = SamplerConfig(temperature=temp, top_k=top_k, top_p=top_p, seed=seed)
        cvk = dict(sampler_config=sampler_config, max_output_tokens=max_output_tokens, **(conv_kw or {}))
        if think: cvk['extra_context'] = {**cvk.get('extra_context', {}), 'enable_thinking': True}
        if filter_think: cvk['filter_channel_content_from_kv_cache'] = True
        self._conv_kw, self.tools, self.conv = cvk, L(tools), None
        self.tool_handler = ChatToolHandler(self)
        self._mk_conv(self._sys_pre + [_to_litert_msg(m) for m in listify(messages)])
        self.ctx_limit = ctx_limit
        self._setup(model=model, sp=sp, messages=messages, tools=tools, approve=approve,
                    tool_max_len=tool_max_len, max_steps=max_steps, final_prompt=final_prompt,
                    cbs=cbs, default_cbs=default_cbs)

    def _mk_conv(self, messages=None):
        "Build the conversation from `messages`, releasing any current one first."
        self._conv_stack.close()
        self._conv_stack = ExitStack()
        self.conv = self._conv_stack.enter_context(self.engine.create_conversation(
            messages=messages or None, tools=list(self.tools) or None,
            tool_event_handler=self.tool_handler, **self._conv_kw))
        return self.conv

    def _recreate_conv(self):
        "Rebuild the `Conversation` from the current (possibly evicted) `hist`, re-applying the system prompt."
        self._mk_conv(self._sys_pre + [_to_litert_msg(m) for m in self.hist])

    def _retry_evicted(self, err, max_output_tokens=None, keep_first=2, keep_last=6):
        "The window filled up mid-turn: evict the middle of `hist`, rebuild, and send the same turn again."
        kept, dropped = evict_middle(self.hist, keep_first, keep_last)
        if not dropped:
            raise ContextWindowExceededError(f"context window full and nothing left to evict: {err}") from err
        self.hist[:] = kept
        self.evicted = getattr(self, 'evicted', 0) + len(dropped)
        # The outgoing message may already be in Python history, but it is about to be
        # sent again. Do not put that copy in the rebuilt engine preface.
        prior = self.hist[:-1] if self._turn_msg_recorded else self.hist
        self._mk_conv(self._sys_pre + [_to_litert_msg(m) for m in prior])
        self._tc0 = self.conv.token_count            # the rebuilt cache is a new baseline for usage
        try: return self.conv.send_message(self.turn_msg, max_output_tokens=max_output_tokens)
        except RuntimeError as e:
            raise ContextWindowExceededError(
                f"could not recover after evicting {len(dropped)} messages: {err}") from err

    @property
    def token_count(self): return self.conv.token_count
    @property
    def cached_tokens(self):
        "Tokens already resident in this conversation's KV cache and reusable by the next turn."
        return self.conv.token_count
    def cancel(self):
        "Cancel the in-flight generation."
        self.conv.cancel_process()
    def count_tokens(self, text):
        "Number of tokens in `text` per the engine tokenizer."
        return len(self.engine.tokenize(text))
    def render(self, msg):
        "The exact templated string litert will send for `msg`."
        return self.conv.render_message_to_string(_mk_msg(msg))
    def _send(self, msg, max_output_tokens=None):
        'Send one message through the callback pipeline, evicting and retrying once if the context window fills up.'
        self.turn_msg = _mk_msg(msg)
        self._turn_msg_recorded = False
        for _ in run_cbs(self, 'before_send'): pass
        self._tc0 = self.conv.token_count      # after the callbacks: one of them may have rebuilt `conv`
        try: r = self.conv.send_message(self.turn_msg, max_output_tokens=max_output_tokens)
        except RuntimeError as e:
            if not is_ctx_error(self, e): raise
            r = self._retry_evicted(e, max_output_tokens)
        self.turn_res = Resp(r)
        for _ in run_cbs(self, 'after_response'): pass
        return self.turn_res
    def _stream(self, msg, max_output_tokens=None, cbs=None):
        'Stream a turn as markdown chunks; per-call `cbs` live only for this turn.'
        added = self.add_cbs(cbs); prev = getattr(self, '_streaming', False); self._streaming = True
        try:
            self.turn_msg = _mk_msg(msg)
            for _ in run_cbs(self, 'before_send'): pass
            self._tc0 = self.conv.token_count   # after the callbacks: one of them may have rebuilt `conv`
            fmt, chunks = StreamFormatter(), []
            for o in self.conv.send_message_async(self.turn_msg, max_output_tokens=max_output_tokens):
                chunks.append(o); yield self._emit(o, fmt)
            self.turn_res = _merge_chunks(chunks)
            yield from run_cbs(self, 'after_response')
            return self.turn_res   # stream's final Resp, captured by SaveReturn / AsyncChat `.value`
        finally: self._streaming = prev; self.remove_cbs(added)
    def _oneshot(self, prompt, sp):
        "Stateless one-shot completion text via a throwaway conversation."
        with self.engine.create_conversation(messages=[{'role': 'system', 'content': sp}] if sp else None) as conv:
            return resp_text(conv.send_message(prompt))
    def _structured_call(self, prompt, schema, sp):
        "Forced tool call; falls back to parsing a JSON reply when the model answers in prose."
        pre = [{'role': 'system', 'content': sp}] if sp else None
        with self.engine.create_conversation(messages=pre, tools=[schema], automatic_tool_calling=False) as conv:
            r = conv.send_message(prompt)
        if tcs := r.get('tool_calls'): return tcs[0].get('function', {}).get('arguments', {})
        try: return json.loads(extract_fence(resp_text(r), 'json'))
        except (json.JSONDecodeError, TypeError) as e: raise ValueError(f"model neither called the tool nor returned JSON; reply: {resp_text(r)[:200]!r}") from e
    def close(self):
        "Release this chat's conversation and, only when owned, its engine (idempotent)."
        if getattr(self, '_conv_stack', None) is not None: self._conv_stack.close(); self._conv_stack = None
        if getattr(self, '_own_engine', False) and getattr(self, '_stack', None) is not None:
            self._stack.close()
        self._stack = None
        self.conv = None

In [ ]:
#| hide
# rebuilding the conversation: a fake engine lets us check what litert would be handed
class _FakeConv:
    def __init__(self, messages=None, **kw): self.messages, self.token_count, self.sent = list(messages or []), 0, []
    def __enter__(self): return self
    def __exit__(self, *a): self.closed = True
    def send_message(self, m, **kw): self.sent.append(m); return {'role': 'assistant', 'content': [{'type': 'text', 'text': 'ok'}]}

class _FakeEngine:
    def __init__(self): self.convs, self.exited = [], False
    def __enter__(self): return self
    def __exit__(self, *a): self.exited = True
    def create_conversation(self, messages=None, **kw):
        c = _FakeConv(messages, **kw); self.convs.append(c); return c
    def tokenize(self, t): return [0] * max(1, len(str(t)) // 4)

eng = _FakeEngine()
chat = LitertChat(engine=eng, sp='anchor me', ctx_limit=100)
test_eq(len(eng.convs), 1)
chat.hist += [{'role': 'user', 'content': f'q{i}'} for i in range(6)]
chat._recreate_conv()
test_eq(len(eng.convs), 2)
assert eng.convs[0].closed                              # the old conversation was released
test_eq(eng.convs[1].messages[0], {'role': 'system', 'content': 'anchor me'})   # sp re-applied
test_eq(len(eng.convs[1].messages), 7)                  # system prompt + the six history messages
chat.close()
assert not eng.exited                              # a caller-owned engine remains usable by sibling chats

# the reactive path: a context error evicts the middle and resends the same turn
class _FullConv(_FakeConv):
    def send_message(self, m, **kw):
        self.sent.append(m)
        if len(self.sent) == 1 and not getattr(self, 'ok', False): raise RuntimeError('max number of tokens reached')
        return {'role': 'assistant', 'content': [{'type': 'text', 'text': 'recovered'}]}

class _FullEngine(_FakeEngine):
    def create_conversation(self, messages=None, **kw):
        c = _FullConv(messages, **kw); c.ok = bool(self.convs); self.convs.append(c); return c

eng2 = _FullEngine()
chat2 = LitertChat(engine=eng2, sp='anchor', ctx_limit=100)
chat2.hist += [{'role': 'user', 'content': f'q{i}'} for i in range(20)]
r = chat2('one more')
test_eq(resp_text(r), 'recovered')
assert chat2.evicted > 0
assert len(eng2.convs) == 2                             # rebuilt exactly once
assert all(str(m) != 'one more' for m in eng2.convs[1].messages)  # resent, not prefixed twice
test_eq([m['content'] for m in chat2.hist][:2], ['q0', 'q1'])

# nothing left to evict -> a typed error rather than a raw litert traceback
eng3 = _FullEngine()
chat3 = LitertChat(engine=eng3, ctx_limit=100)
test_fail(lambda: chat3('hi'), contains='nothing left to evict')

In [ ]:
# fmt2hist / hist2fmt round-trip (model-free)
_nat = [
    'hi',
    Message.model(Contents.of('hello')).to_json(),
    {'role': 'model', 'tool_calls': [{'type': 'function', 'function': {'name': 'add', 'arguments': {'a': 2, 'b': 3}}}]},
    {'role': 'tool', 'content': [{'type': 'tool_response', 'name': 'add', 'response': 5}]},
]
_canon = LitertChat.fmt2hist(_nat)
test_eq(_canon[0], {'role': 'user', 'content': 'hi'})
test_eq(_canon[1], {'role': 'assistant', 'content': 'hello'})
test_eq(_canon[2]['role'], 'assistant')
test_eq(_canon[2]['tool_calls'][0]['function'], {'name': 'add', 'arguments': {'a': 2, 'b': 3}})
assert _canon[2]['tool_calls'][0]['id']                       # a synthetic id was assigned
test_eq(_canon[3], {'role': 'tool', 'name': 'add', 'content': '5'})

# canonical -> litert Messages ready for the engine
_ms = LitertChat.hist2fmt(_canon)
test_eq([m.role.value for m in _ms], ['user', 'model', 'model', 'tool'])
test_eq(str(_ms[1]), 'hello')
test_eq(_ms[2].tool_calls[0].name, 'add')
test_eq(_ms[3].contents.to_json(), [{'type': 'tool_response', 'name': 'add', 'response': '5'}])

# fmt2hist is idempotent on an already-canonical (llama-origin) tool round
_llama = [{'role': 'assistant', 'content': '', 'tool_calls': [{'id': 'call_x', 'type': 'function',
                                                               'function': {'name': 'add', 'arguments': {'a': 1}}}]},
          {'role': 'tool', 'tool_call_id': 'call_x', 'name': 'add', 'content': '1'}]
test_eq(LitertChat.fmt2hist(_llama), _llama)

### Managing callbacks

`add_cb` registers a callback and returns the instance; `add_cbs` takes a list and returns the instances. `remove_cb` drops one by instance, or by class to remove every callback of that type, and `remove_cbs` handles several at once. To run a callback for a single turn, pass `cbs=` to the call: `chat(msg, cbs=[PyFenceCallback(...)])` registers them before the turn and removes them after, so nothing leaks into later turns.

In [ ]:
#| eval: false
set_min_log_severity(5)
_get_model(gemma4_12b)

'/Users/71293/.cache/huggingface/hub/models--litert-community--gemma-4-12B-it-litert-lm/snapshots/44cf85a326f79b814fa86a60af414c042755b43a/gemma-4-12B-it.litertlm'

## Human-in-the-loop tool approval

Any `approve(tool_call) -> bool` passed to `Chat(approve=...)` is consulted by `ChatToolHandler` before each tool runs. `hitl_policy` builds one from a per-tool policy: `'approved'` (auto-run), `'dont_run'` (always block), or `'check'` (ask, via `_ask_console` by default). Or supply your own function for custom logic such as logging, rate-limiting, or prompting a UI. See the worked example below.

## Using Chat

A short tour of the main features. These cells build a real model, so they are set to run manually rather than in the test suite.

In [ ]:
#| eval: false
chat=Chat(backend=Backend.GPU(), cache_dir='.cache/litertlm', think=True)
# chat_12 = LitertChat(engine=LitertChat.create_engine(gemma4_12b, multimodal=False, cache_dir='.cache/litertlm', be=Backend.GPU()))

In [ ]:
#| eval: false
# set_min_log_severity(2)
r = chat("Reply with exactly: pong")
assert 'pong' in resp_text(r).lower()
assert resp_text(chat.hist[-1]) == resp_text(r) and chat.hist[0]['role'] == 'user'
assert chat.use.total_tokens > 0 and chat.token_count > 0
print(chat.use); chat.print_hist()

total=104|in=103|out=1|turns=1


**user**

Reply with exactly: pong

---

**assistant**

> **🧠 Thinking**
>
> Thinking Process:
> 
> 1.  **Analyze the Request:** The user has instructed me to reply with *exactly* the word "pong".
> 2.  **Determine the Constraint:** The constraint is strict: "Reply with exactly: pong".
> 3.  **Formulate the Response:** The response must be the string "pong".
> 4.  **Final Output Generation:** pong

pong

In [ ]:
#| eval: false
# Real LiteRT conversation-cache test. BenchmarkInfo reports the actual last prefill performed by
# the runtime; turn two should prefill only its short new message while retaining the old KV context.
kvchat = Chat(backend=Backend.GPU(), cache_dir='.cache/litertlm', think=False, max_output_tokens=16,
              eng_kw={'enable_benchmark': True})
try:
    kvchat(('cache verification context ' * 96) + '\nReply with exactly: stored')
    first = kvchat.conv.get_benchmark_info()
    context_before_turn2 = kvchat.token_count
    r = kvchat('What exact word did I ask you to reply with? Reply with only that word.')
    second = kvchat.conv.get_benchmark_info()
    print(dict(first_prefill=first.last_prefill_token_count,
               second_prefill=second.last_prefill_token_count,
               context_before_turn2=context_before_turn2,
               context_after_turn2=kvchat.token_count,
               reported_cached=kvchat.use.cached_tokens,
               answer=resp_text(r)))
    assert context_before_turn2 > 0 and kvchat.token_count > context_before_turn2
    assert second.last_prefill_token_count < context_before_turn2, 'turn two appears to have re-prefilled the old context'
    assert kvchat.use.cached_tokens == context_before_turn2, kvchat.use
    assert 'stored' in resp_text(r).lower(), resp_text(r)
finally:
    kvchat.close()


{'first_prefill': 304, 'second_prefill': 25, 'context_before_turn2': 306, 'context_after_turn2': 333, 'reported_cached': 306, 'answer': 'stored'}


In [ ]:
#| eval: false
chat(['what is pong.','do you know?'])

### Images and audio

Some Gemma builds accept image and audio content alongside text.

In [ ]:
#| eval: false
from PIL import Image

In [ ]:
#| eval: false
im=Image.open(Path.cwd()/'nbs'/'images.jpeg');im

In [ ]:
#| eval: false
chat(['explain this image', img_bytes(im)])
# you can also use ImageFile and ImageBytes

In [ ]:
#| eval: false
chat(['transcribe this audio', AudioFile(str(Path.cwd()/'nbs'/'speech.wav'))])

### Example: gating a tool behind approval

A custom `approve` gate that auto-allows a safe tool and blocks a destructive one. `ChatToolHandler` routes every tool call through it; a denied call is recorded in history as `Denied by human operator` and reported back to the model, which continues without it.

In [ ]:
#| eval: false
def add(a: int, b: int) -> int:
    'Add two integers.\n\nArgs:\n    a: first addend\n    b: second addend'
    return a + b

def delete_files(path: str) -> str:
    'Delete everything under a path.\n\nArgs:\n    path: directory to wipe'
    return f"wiped {path}"

def approve_gate(tc):
    "chat.approve: allow any tool except destructive ones; log every decision."
    name = tc['function']['name']
    ok = name != 'delete_files'
    print(f"[approval] {tc_summary_(name, tc['function'].get('arguments', {}))} -> {'ALLOW' if ok else 'DENY'}")
    return ok

chat = Chat(tools=[add, delete_files], approve=approve_gate, sp="Use the tools to satisfy the request.")
r = chat("Add 2 and 3, then delete everything under /tmp/data.")
print('\n', resp_text(r))
chat.print_hist()   # the blocked call is logged as a 'Denied by human operator' tool response
chat.close()

# Declarative equivalent (interactive 'check' prompts on the console via _ask_console):
# Chat(tools=[add, delete_files], approve=hitl_policy({'add': 'approved', 'delete_files': 'dont_run'}))

### Policy-driven approval

You can also drive approvals declaratively with `hitl_policy` instead of writing your own function.

In [ ]:
#| eval: false
chat=Chat(tools=[add, delete_files], approve=hitl_policy({'add': 'approved', 'delete_files': 'check'}, browser=True))

In [ ]:
#| eval: false
r = chat("Add 2 and 3, then delete everything under /tmp/data.")
print('\n', resp_text(r))


In [ ]:
chat.print_hist()   # the blocked call is logged as a 'Denied by human operator' tool response
chat.close()

## Streaming turns

Pass `stream=True` and iterate the result: `Chat` returns a generator of markdown chunks (formatted by `StreamFormatter`) instead of a response dict. History and usage are finalised once the stream is exhausted.

In [ ]:
#| eval: false
ch = Chat(backend=Backend.GPU())

In [ ]:
#| eval: false
# stream chunks to a terminal:
for c in ch("Count: one two three", stream=True): print(c, end='', flush=True)

That's a simple count!

If you'd like me to do something with that count, please let me know. For example, would you like me to:

* **Continue counting?** (e.g., "four, five, six...")
* **Count something else?**
* **Do a math problem?**
* **Something else entirely?**

In [ ]:
#| eval: false
# sync streaming: SaveReturn wraps the chunk stream; display_stream renders live + returns md, `.value` is the final Resp
from fastcore.xtras import SaveReturn
s = SaveReturn(ch('count fibonacci for 10 places', stream=True))
md = display_stream(s)
assert md.strip() and resp_text(s.value) == resp_text(ch.turn_res)

Here are the first 10 numbers in the Fibonacci sequence:

The Fibonacci sequence starts with 0 and 1, and each subsequent number is the sum of the two preceding ones.

1. **0**
2. **1**
3. **1** (0 + 1)
4. **2** (1 + 1)
5. **3** (1 + 2)
6. **5** (2 + 3)
7. **8** (3 + 5)
8. **13** (5 + 8)
9. **21** (8 + 13)
10. **34** (13 + 21)

**The first 10 Fibonacci numbers are: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34**

'Here are the first 10 numbers in the Fibonacci sequence:\n\nThe Fibonacci sequence starts with 0 and 1, and each subsequent number is the sum of the two preceding ones.\n\n1. **0**\n2. **1**\n3. **1** (0 + 1)\n4. **2** (1 + 1)\n5. **3** (1 + 2)\n6. **5** (2 + 3)\n7. **8** (3 + 5)\n8. **13** (5 + 8)\n9. **21** (8 + 13)\n10. **34** (13 + 21)\n\n**The first 10 Fibonacci numbers are: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34**'

In [ ]:
#| eval: false
# async streaming: adisplay_stream drives the turn; the returned iterator carries the final Resp on `.value`
achat = AsyncChat(Chat(cache_dir='.cache/litertlm'))
sa = run_coro(achat('Count: one two three', stream=True))
md = run_coro(adisplay_stream(sa))
assert md.strip() and resp_text(sa.value) == resp_text(achat.turn_res)
assert [m['role'] for m in achat.hist] == ['user', 'assistant']
achat.close()

In [ ]:
#| eval: false
assert resp_text(chat.hist[-1]) == resp_text(chat.turn_res) and chat.use.completion_tokens > 0

In [ ]:
#| eval: false
fired = []
class Log(ChatCallback):
    def before_tool_calls(self): fired.append('before')
    def after_tool_calls(self):  fired.append('after')

chat = Chat(tools=[add], sp="Use the add tool for arithmetic.")
chat.add_cb(Log)
r = chat("What is 21 + 21? Use the tool.")
print(fired)
assert 'before' in fired and 'after' in fired
assert any(m.get('role') == 'tool' for m in chat.hist)
print(resp_text(r)); chat.print_hist()

['before', 'after']
The result of adding 21 and 21 is 42.


**user**

What is 21 + 21? Use the tool. 

---

**model**



🔧 add({'a': 21.0, 'b': 21.0})

---

**tool**



↩︎ **add**: 42.0

---

**assistant**

The result of adding 21 and 21 is 42.

In [ ]:
#| hide
merged = _merge_chunks([{"content": [{"type": "text", "text": "po"}]},
                        {"content": [{"type": "text", "text": "ng"}]}])
test_eq(merged, {"role": "assistant", "content": [{"type": "text", "text": "pong"}]})

## Running python from replies

`PyFenceCallback` turns the chat into a code interpreter. After a reply it finds the last ` ```python ` fence, runs it through `safepyrun` (sandboxed, so imports like `socket` and `importlib` are blocked) in a namespace that persists across the conversation, feeds the output back as a ` ```result ` block, and re-queries, up to `max_rounds` times. Enable it with `Chat(cbs=[PyFenceCallback])`. Execution goes through the same `approve` hook as a synthetic `python` tool, so `hitl_policy({'python': 'check'})` prompts before running. `chat.run_py(code)` runs a snippet directly (`ban_defs=False` allows `def` and `class`).

### Stopping the loop with `done`

Pass a `done` function to `PyFenceCallback` to end the loop early. `output_matches` stops once the code output contains an expected string.

In [ ]:
#| eval: false
chat = Chat(sp="When asked to compute something, reply with a ```python fence that prints or evaluates the answer.")
r = chat("Use python to compute 2**5.", cbs=[PyFenceCallback(done=output_matches(32))])
chat.print_hist()   # you'll see the ```python turn, a ```result turn, then the final answer
chat.close()

**user**

Use python to compute 2**5. 

---

**assistant**

```python
print(2**5)
```

---

**user**

32


### Letting the model decide

Or leave termination to the model: with no `done`, the loop stops as soon as a reply has no code fence.

In [ ]:
#| eval: false
chat = Chat(cbs=[PyFenceCallback], sp="When asked to compute something, reply with a ```python fence that prints or evaluates the answer.")
r = chat("Use python to compute 2**5.")
chat.print_hist()
chat.close()

**user**

Use python to compute 2**5. 

---

**assistant**

```python
print(2**5)
```

---

**user**

```result
32

```

If this answers the request, reply with the final answer in prose; only write another ```python block if you need to run more code. 

---

**assistant**

32

## Utilities: classify, structured, benchmark

`classify` and `structured` each run one-shot in a throwaway conversation on the shared engine, isolated from the live chat, so they leave its history and KV cache untouched. `classify` generates a label and matches it against your options. `structured` forces a tool call and returns `schema(**arguments)`. `bench` reports init time, time to first token, and prefill and decode tokens per second.

litert's `run_text_scoring` log-likelihood scoring is not available on this runtime, so `classify` generates and label-matches rather than scoring.

In [ ]:
#| export
def bench(model_id=gemma4_e2b, model_path=None, backend=Backend.CPU(), prefill_tokens=64, decode_tokens=64, **kw):
    "Benchmark init time, TTFT, and prefill/decode tokens-per-sec via litert's `Benchmark`."
    return Benchmark(_get_model(model_id, model_path), backend=backend, prefill_tokens=prefill_tokens, decode_tokens=decode_tokens, **kw).run()


In [ ]:
#| eval: false
# classify + structured on a real model
sm = Chat(cache_dir='.cache/litertlm')
test_eq(sm.classify("I absolutely loved this film!", ['positive', 'negative']), 'positive')

from dataclasses import dataclass
@dataclass
class Person: name:str; age:int
p = sm.structured("Extract the person: John Smith is 30 years old.", Person)
print(p); assert isinstance(p, Person) and 'John' in p.name
sm.close()

Person(name='John Smith', age=30)


## Grading answers

`chat.check` turns the model into a graded question-answerer. It asks `question` in an isolated conversation (like `classify`/`structured`, so nothing touches `chat.hist`), pulls the model's answer out of a ```answer fence with `extract_fence` (falling back to the whole reply if the model skips the fence), and grades it against `expected`. Grading is `grade_fn(answer, expected) -> bool`, defaulting to `matches_` (the answer must contain `expected`, or any value in an `expected` list). Pass your own `grade_fn` for custom logic, or set `llm_judge=True` to grade with the model instead (`chat.grades`, built on `classify`); pass `judge=` a second `Chat` to have a stronger model do the grading.

In [ ]:
#| eval: false
# Default grade is a deterministic match (no judge needed):
qa = Chat(cache_dir='.cache/litertlm')
print(qa.check("What is the capital of France?", "Paris").ok)
# Custom grader: any `answer, expected -> bool`:
print(qa.check("What is 2 + 2?", "4", grade_fn=lambda a, e: e in a).ok)

True
True


In [ ]:
#| eval: false
# LLM-as-judge, optionally with a bigger model doing the grading:
judge = Chat(gemma4_12b, backend=Backend.GPU(), multimodal=False, cache_dir='.cache/litertlm')

In [ ]:
#| eval: false
r = qa.check("Name a primary colour.", "red, blue, or yellow", judge=judge)   # judge -> llm_judge
print(r.answer, '->', r.ok)
judge.close(); qa.close()

Red -> True


In [ ]:
#| eval: false
# grades (LLM judge) + check on a real model
sm = Chat(cache_dir='.cache/litertlm')
assert sm.grades("What is the capital of France?", "Paris", "Paris") is True
assert sm.grades("What is the capital of France?", "Paris", "London") is False
r = sm.check("What is the capital of France?", "Paris")            # deterministic default
print(r.answer, '->', r.ok); assert r.ok
assert sm.check("What is 2 + 2?", "4", llm_judge=True).ok is True  # self as judge
sm.close()

Paris -> True


## Installing the skill

`skill.md` ships with the package. A harness can drop it into the standard skill directories with `mv_skill_md`, which writes `SKILL.md` under `.claude/skills/rishi/` and `.agents/skills/rishi/` at the git root. It is a dry run by default and only prints the targets; pass `dry_run=False` to write, or `dir=` to install somewhere else.

In [ ]:
from fastcore.test import test_eq
import rishi.core, rishi.litert
test_eq(rishi.core.get_runtime('litert'), rishi.litert.LitertChat)

# dispatch really happens in `Chat.__new__`, so it can be checked without loading a model
test_eq(type(rishi.core.Chat.__new__(rishi.core.Chat)), rishi.litert.LitertChat)              # the default runtime
test_eq(type(rishi.core.Chat.__new__(rishi.core.Chat, 'litert-community/gemma-4-E2B-it-litert-lm')),
        rishi.litert.LitertChat)
test_eq(rishi.litert.LitertChat._runtime, 'litert')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()